In [1]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from tqdm import tqdm
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
import matplotlib as mpl
pd.options.mode.chained_assignment = None


eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_p2d/"
res_DIR = "../data/results_p2d/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [2]:
cell = 1

In [3]:
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)

In [4]:
sno = 0

# DFN

In [5]:
N

array([  0,  18,  57,  93, 134, 175, 216, 257, 298, 339], dtype=int64)

In [6]:
spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
        # "particle mechanics":"swelling only",
    }
)
param=spm.param
parameter_values = get_parameter_values()   
# sim_des = sim_des+'_cv'
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
Ns = np.insert(N_0[1:]-1,0,0)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe,spm,parameter_values)
pybamm.set_logging_level("WARNING")
par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
parameter_values = get_parameter_values()
parameter_values.update(
    {   
        "Positive electrode diffusion coefficient activation energy [J.mol-1]": 0,
        "Negative electrode diffusion coefficient activation energy [J.mol-1]": 0,
        "Positive electrode reference exchange-current density activation energy [J.mol-1]": 0,
        "Negative electrode reference exchange-current density activation energy [J.mol-1]": 0,
        "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        "Positive electrode reference exchange-current density [A.m-2(m3.mol)1.5]": 3.377e-06,
        "Negative electrode reference exchange-current density [A.m-2(m3.mol)1.5]":	3.183e-06,
        "Negative electrode active material volume fraction": eps_n_data,
        "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "Positive electrode LAM constant proportional term [s-1]": 0*par_val[sno][0],
        "Negative electrode LAM constant proportional term [s-1]": 0*par_val[sno][1],
        "Positive electrode LAM constant proportional term 2 [s-1]": 0*par_val[sno][5],
        "Negative electrode LAM constant proportional term 2 [s-1]": 0*par_val[sno][4],
        "Positive electrode LAM constant exponential term": par_val[sno][2],
        "Negative electrode LAM constant exponential term": par_val[sno][2],
        "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Lithium plating kinetic rate constant [m.s-1]": 0*par_val[sno][3],
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "Li plating resistivity [Ohm.m]": 30000,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,
        # "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        # "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        # "Negative electrode critical stress [Pa]": 20e+06,
        # "Positive electrode critical stress [Pa]": 40e+06,
    },
    check_already_exists=False,
)
experiment = pybamm.Experiment(
        [
            ("Discharge at "+c_rate_d+dis_set,
            "Rest for 10 sec",
            "Charge at "+c_rate_c+" until 4.2V", 
            "Hold at 4.2V until C/100")
        ] *1,
        termination="20% capacity",
    #     cccv_handling="ode",
    )
# all_sumvars_dict = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)
# with open('spm_sim_sei_c5_eSOH.pickle', 'wb') as handle:
#     pickle.dump(all_sumvars_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)
sim_long = pybamm.Simulation(spm, experiment=experiment, parameter_values=parameter_values, 
                                solver=pybamm.CasadiSolver("safe"))
sol1 = sim_long.solve(initial_soc=SOC_0)
sols = []
sols.append(sol1)
for i in tqdm(range((dfe.N.iloc[-1])-1)):
    spm.set_initial_conditions_from(sol1, inplace=True)
    sim_long = pybamm.Simulation(spm, experiment=experiment, parameter_values=parameter_values, 
                                    solver=pybamm.CasadiSolver("safe"))
    sol1 = sim_long.solve()
    sols.append(sol1)

100%|██████████| 338/338 [13:39<00:00,  2.42s/it]


In [13]:
def get_eSOH(sols):
    n_Li_a = []
    x_0_a = []
    x_100_a = []
    y_0_a = []
    y_100_a = []
    delta_sei_a = []
    time_a = []
    Qmax_a = []
    Ahth_a = []
    C_a = []
    time = 0
    Ahth = 0
    for i in range(len(sols)):
        sol = sols[i]
        sum_var = sol.summary_variables
        del_sei = sum_var["X-averaged SEI thickness [m]"][0]
        C = sum_var["C"][0]
        x_0 = sum_var["x_0"][0]
        x_100 = sum_var["x_100"][0]
        y_0 = sum_var["y_0"][0]
        y_100 = sum_var["y_100"][0]
        n_Li = sum_var["n_Li"][0]
        t = sol["Time [s]"].entries
        Q = sol["Discharge capacity [A.h]"].entries
        Qmax = max(abs(Q))
        Ahth += np.cumsum(abs(np.diff(Q)))[-1]
        time += t[-1]
        x_0_a.append(x_0)
        x_100_a.append(x_100)
        y_0_a.append(y_0)
        y_100_a.append(y_100)
        n_Li_a.append(n_Li)
        delta_sei_a.append(del_sei)
        time_a.append(time)
        Qmax_a.append(Qmax)
        Ahth_a.append(Ahth)
        C_a.append(C)
    
    return (x_100_a,y_100_a,x_0_a,y_0_a,n_Li_a,delta_sei_a,C_a,time_a,Qmax_a,Ahth_a)

In [10]:
Q = sol1["Discharge capacity [A.h]"].entries

In [14]:
x_100_a,y_100_a,x_0_a,y_0_a,n_Li_a,delta_sei_a,C_a,time_a,Qmax_a,Ahth_a = get_eSOH(sols)

In [15]:
with open('spm_sim_sei_c5_eSOH_V2.pickle', 'wb') as handle:
        pickle.dump((x_100_a,y_100_a,x_0_a,y_0_a,n_Li_a,delta_sei_a,C_a,time_a,Qmax_a,Ahth_a), handle, protocol=pickle.HIGHEST_PROTOCOL)